# Task 2 Baseline: Separate Conditional Bernoulli MLP per Recording

This notebook follows the style of `Final_Task_Example.ipynb`, but changes the task from decoding `x -> y` to encoding `z -> x`.

Important design choice for this version:

- `group_data = True`
- A **separate model is trained for each `(network, DIV)` dataset**.
- The model input does **not** include network ID or DIV.
- Each model only conditions on the stimulation condition available within that dataset: stimulation pattern and train frequency.

This lets you inspect whether different recordings produce meaningfully different learned mappings. If the separate models behave similarly, then it is reasonable to try a pooled single-dataset model later.


# 1. Prerequisites

In [ ]:
from utils.data import load_data, save_data
from utils.plotting import create_post_stim_raster_plot, map_colour_to_electrode

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
import getpass
from pathlib import Path

In [ ]:
# Group 4 assigned datasets from the table.
# The table's stimulation/protocol column is intentionally ignored here.
# Each network/DIV pair is treated as its own independent dataset.
Assigned_Recordings = [
    {"network": 5, "div": 21},
    {"network": 5, "div": 30},
    {"network": 6, "div": 21},
    {"network": 6, "div": 30},
    {"network": 7, "div": 21},
    {"network": 7, "div": 30},
    {"network": 8, "div": 21},
]

group_data = True
test_data = False

# Set to a subset while debugging, e.g.:
# Assigned_Recordings = [{"network": 5, "div": 21}]

# 2. Model and Feature Definitions

In [ ]:
def make_condition_features(
    stimulation_parameters,
    n_classes=16,
    freq_mean=None,
    freq_std=None,
):
    """
    Build z features for Task 2.

    Expected stimulation_parameters style from the example notebook:
      column 0: stimulation frequency
      column 1: stimulation pattern index

    Features used in this separate-recording version:
      - 16-dimensional one-hot stimulation pattern
      - normalized log-frequency

    Network and DIV are deliberately NOT included as inputs.
    Because each model is trained for one dataset only, recording identity is implicit in the model checkpoint.
    """
    stimulation_parameters = np.asarray(stimulation_parameters)
    frequencies = stimulation_parameters[:, 0].astype(np.float32)
    patterns = stimulation_parameters[:, 1].astype(np.int64)

    # Robustness: convert 1..16 labels to 0..15 if needed.
    if patterns.min() == 1 and patterns.max() == n_classes:
        patterns = patterns - 1

    if patterns.min() < 0 or patterns.max() >= n_classes:
        raise ValueError(
            f"Pattern labels must be in 0..{n_classes-1} or 1..{n_classes}; "
            f"got min={patterns.min()}, max={patterns.max()}."
        )

    # Frequencies span a broad range, so log scaling is safer than raw Hz.
    log_freq = np.log1p(frequencies).astype(np.float32)
    if freq_mean is None:
        freq_mean = log_freq.mean()
    if freq_std is None:
        freq_std = log_freq.std() + 1e-6
    log_freq = ((log_freq - freq_mean) / freq_std)[:, None]

    pattern_one_hot = F.one_hot(torch.as_tensor(patterns).long(), num_classes=n_classes).float().numpy()
    z = np.concatenate([pattern_one_hot, log_freq], axis=1).astype(np.float32)
    return z, freq_mean, freq_std


class ConditionalBernoulliMLP(nn.Module):
    """
    Baseline encoder for Task 2: z -> expected response raster.

    Input:  condition vector z = [pattern one-hot, normalized log-frequency]
    Output: logits with shape (Batch, 1, N, T)
    """

    def __init__(self, n_inputs, response_shape, n_hidden=256, dropout=0.20):
        super().__init__()
        self.response_shape = response_shape
        n_outputs = int(np.prod(response_shape))

        self.fully_connected_layers = nn.Sequential(
            nn.Linear(n_inputs, n_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(n_hidden, n_hidden * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(n_hidden * 2, n_outputs),
        )

    def forward(self, z):
        x = self.fully_connected_layers(z)
        x = x.view(z.shape[0], *self.response_shape)
        return x


def spike_count_mae_from_probs(probs, target):
    pred_counts = probs.flatten(1).sum(dim=1)
    true_counts = target.flatten(1).sum(dim=1)
    return torch.abs(pred_counts - true_counts)

# 3. Training Function for One Recording

In [ ]:
# Hyperparameters
n_epochs = 200
batch_size = 256
learning_rate = 1e-4
val_frac = 0.1
n_classes = 16
random_seed = 0

# Keep checkpoints in the same home-directory style as the example notebook.
checkpoint_dir = Path(f"/home/{getpass.getuser()}")

def train_one_recording(rec):
    """Train and evaluate one independent Task 2 baseline model for one network/DIV dataset."""
    network = rec["network"]
    div = rec["div"]
    rec_name = f"network{network}_DIV{div}"
    checkpoint_path = checkpoint_dir / f"best_task2_baseline_{rec_name}.pth"

    print("=" * 80)
    print(f"Training separate Task 2 baseline for {rec_name}")
    print("=" * 80)

    # Load one dataset only.
    data = load_data(network, div, group_data, test_mode=False)
    stimulation_parameters, stimulation_patterns, responses, stimulation_times, impedance_map, electrodes = data

    print(f"responses: {responses.shape}")
    print(f"stimulation_parameters: {stimulation_parameters.shape}")

    # Keep response in (Batch, Channel, Electrode, Time) format, matching the example notebook style.
    if responses.ndim != 4:
        print(f"Initial response size is {responses.shape}")
        responses = responses[:, None]
        print(f"Adjusted response size is {responses.shape}")

    # Random trial split within this recording only.
    rng = np.random.default_rng(random_seed)
    idx = np.arange(responses.shape[0])
    rng.shuffle(idx)
    n_val = max(1, int(len(idx) * val_frac))
    val_idx = idx[:n_val]
    trn_idx = idx[n_val:]

    Xtr = responses[trn_idx]
    Ptr = stimulation_parameters[trn_idx]
    Xval = responses[val_idx]
    Pval = stimulation_parameters[val_idx]

    # Fit frequency normalization on this recording's training split only.
    Ztr, freq_mean, freq_std = make_condition_features(Ptr, n_classes=n_classes)
    Zval, _, _ = make_condition_features(Pval, n_classes=n_classes, freq_mean=freq_mean, freq_std=freq_std)

    response_shape = tuple(Xtr.shape[1:])
    n_inputs = Ztr.shape[1]
    print(f"Condition feature size: {n_inputs}")
    print(f"Response shape predicted by model: {response_shape}")
    print("Network/DIV are not model inputs in this version.")

    # Weighted BCE helps prevent all-zero predictions on sparse spike bins.
    pos_count = float(Xtr.sum())
    neg_count = float(Xtr.size - Xtr.sum())
    pos_weight_value = neg_count / max(pos_count, 1.0)
    pos_weight_value = min(pos_weight_value, 50.0)
    print(f"Positive-bin weight used in BCE: {pos_weight_value:.2f}")

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight_value, dtype=torch.float32),
        reduction="none",
    )

    model = ConditionalBernoulliMLP(
        n_inputs=n_inputs,
        response_shape=response_shape,
        n_hidden=256,
        dropout=0.20,
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    print(f"The model has {sum(p.numel() for p in model.parameters())} parameters.")

    train_dataset = TensorDataset(
        torch.as_tensor(Ztr, dtype=torch.float32),
        torch.as_tensor(Xtr, dtype=torch.float32),
        torch.as_tensor(Ptr, dtype=torch.float32),
    )
    val_dataset = TensorDataset(
        torch.as_tensor(Zval, dtype=torch.float32),
        torch.as_tensor(Xval, dtype=torch.float32),
        torch.as_tensor(Pval, dtype=torch.float32),
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion.to(device)

    best_val_loss = float("inf")
    train_losses = []
    val_losses_by_epoch = []
    val_count_mae_by_epoch = []

    pbar = tqdm(range(n_epochs), leave=False)
    for epoch in pbar:
        model.train()
        running_train_loss = 0.0

        for z, x, p in train_loader:
            z, x = z.to(device), x.to(device)
            optimizer.zero_grad()
            logits = model(z)
            loss = torch.mean(criterion(logits, x))
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item()

        train_loss = running_train_loss / len(train_loader)
        train_losses.append(train_loss)

        model.eval()
        val_loss = []
        val_mae = []
        with torch.no_grad():
            for z, x, p in val_loader:
                z, x = z.to(device), x.to(device)
                logits = model(z)
                loss_per_sample = criterion(logits, x).flatten(1).mean(dim=1)
                probs = torch.sigmoid(logits)
                count_mae = spike_count_mae_from_probs(probs, x)
                val_loss.append(loss_per_sample.numpy(force=True))
                val_mae.append(count_mae.numpy(force=True))

        val_loss = np.concatenate(val_loss)
        val_mae = np.concatenate(val_mae)
        avg_val_loss = float(val_loss.mean())
        avg_val_mae = float(val_mae.mean())
        val_losses_by_epoch.append(avg_val_loss)
        val_count_mae_by_epoch.append(avg_val_mae)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), checkpoint_path)

        pbar.set_postfix(
            loss_train=f"{train_loss:.4f}",
            loss_val=f"{avg_val_loss:.4f}",
            count_mae=f"{avg_val_mae:.2f}",
            best_loss_val=f"{best_val_loss:.4f}",
        )

    # Evaluate best checkpoint using unweighted BCE for model-vs-mean comparison.
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True, map_location=device))
    model.eval()

    mean_response = train_dataset.tensors[1].mean(dim=0, keepdim=True).to(device)

    model_bce = []
    mean_bce = []
    model_count_mae = []
    mean_count_mae = []
    model_predictions = []

    with torch.no_grad():
        for z, x, p in val_loader:
            z, x = z.to(device), x.to(device)
            logits = model(z)
            probs = torch.sigmoid(logits)
            mean_probs = mean_response.expand_as(x).clamp(1e-6, 1 - 1e-6)

            model_bce.append(F.binary_cross_entropy(probs, x, reduction="none").flatten(1).mean(dim=1).numpy(force=True))
            mean_bce.append(F.binary_cross_entropy(mean_probs, x, reduction="none").flatten(1).mean(dim=1).numpy(force=True))
            model_count_mae.append(spike_count_mae_from_probs(probs, x).numpy(force=True))
            mean_count_mae.append(spike_count_mae_from_probs(mean_probs, x).numpy(force=True))
            model_predictions.append(probs.numpy(force=True))

    model_bce = np.concatenate(model_bce)
    mean_bce = np.concatenate(mean_bce)
    model_count_mae = np.concatenate(model_count_mae)
    mean_count_mae = np.concatenate(mean_count_mae)
    model_predictions = np.concatenate(model_predictions, axis=0)

    result = {
        "recording": rec,
        "name": rec_name,
        "checkpoint_path": str(checkpoint_path),
        "freq_mean": float(freq_mean),
        "freq_std": float(freq_std),
        "response_shape": response_shape,
        "n_inputs": n_inputs,
        "train_losses": np.array(train_losses),
        "val_losses_by_epoch": np.array(val_losses_by_epoch),
        "val_count_mae_by_epoch": np.array(val_count_mae_by_epoch),
        "model_bce": model_bce,
        "mean_bce": mean_bce,
        "model_count_mae": model_count_mae,
        "mean_count_mae": mean_count_mae,
        "Pval": Pval,
        "Xval": Xval,
        "predictions_val": model_predictions,
        "stimulation_parameters": stimulation_parameters,
        "impedance_map": impedance_map,
        "electrodes": electrodes,
    }

    print(f"{rec_name} model validation BCE:      {model_bce.mean():.5f}")
    print(f"{rec_name} mean baseline BCE:         {mean_bce.mean():.5f}")
    print(f"{rec_name} model spike-count MAE:     {model_count_mae.mean():.2f}")
    print(f"{rec_name} mean baseline count MAE:   {mean_count_mae.mean():.2f}")
    print(f"Saved checkpoint: {checkpoint_path}")

    return result

# 4. Train One Model per `(network, DIV)` Recording

In [ ]:
results = []
for rec in Assigned_Recordings:
    results.append(train_one_recording(rec))

print("Finished training separate models for all assigned recordings.")

# 5. Compare Recordings

In [ ]:
# Summary table printed as plain output to keep the notebook dependency-light.
print("Recording-level validation summary")
print("name\tmodel_BCE\tmean_BCE\tmodel_count_MAE\tmean_count_MAE")
for r in results:
    print(
        f"{r['name']}\t"
        f"{r['model_bce'].mean():.5f}\t"
        f"{r['mean_bce'].mean():.5f}\t"
        f"{r['model_count_mae'].mean():.2f}\t"
        f"{r['mean_count_mae'].mean():.2f}"
    )

In [ ]:
recording_names = [r["name"] for r in results]
model_bce_means = np.array([r["model_bce"].mean() for r in results])
mean_bce_means = np.array([r["mean_bce"].mean() for r in results])
model_count_mae_means = np.array([r["model_count_mae"].mean() for r in results])
mean_count_mae_means = np.array([r["mean_count_mae"].mean() for r in results])

x_axis = np.arange(len(recording_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x_axis - width / 2, model_bce_means, width, label="MLP")
ax.bar(x_axis + width / 2, mean_bce_means, width, label="within-recording mean")
ax.set_xticks(x_axis)
ax.set_xticklabels(recording_names, rotation=45, ha="right")
ax.set_ylabel("BCE")
ax.set_title("Task 2 validation BCE by separate recording model")
ax.legend()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x_axis - width / 2, model_count_mae_means, width, label="MLP")
ax.bar(x_axis + width / 2, mean_count_mae_means, width, label="within-recording mean")
ax.set_xticks(x_axis)
ax.set_xticklabels(recording_names, rotation=45, ha="right")
ax.set_ylabel("Spike-count MAE")
ax.set_title("Task 2 spike-count MAE by separate recording model")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Training curves for each separate model.
for r in results:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, len(r["train_losses"]) + 1), r["train_losses"], label="train weighted BCE")
    ax.plot(range(1, len(r["val_losses_by_epoch"]) + 1), r["val_losses_by_epoch"], label="validation weighted BCE")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Weighted BCE")
    ax.set_title(f"Training curve: {r['name']}")
    ax.legend()
    fig.tight_layout()
    plt.show()

# 6. Within-Recording Diagnostics

In [ ]:
# Per-frequency and per-pattern diagnostics for each separately trained model.
for r in results:
    Pval = r["Pval"]
    frequencies = Pval[:, 0]
    patterns = Pval[:, 1]

    unique_freqs = np.unique(frequencies)
    bce_per_freq = np.array([r["model_bce"][frequencies == f].mean() for f in unique_freqs])
    mae_per_freq = np.array([r["model_count_mae"][frequencies == f].mean() for f in unique_freqs])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(unique_freqs, bce_per_freq, marker="o", markersize=3)
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel("BCE")
    ax.set_title(f"BCE per frequency: {r['name']}")
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(unique_freqs, mae_per_freq, marker="o", markersize=3)
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel("Spike-count MAE")
    ax.set_title(f"Spike-count MAE per frequency: {r['name']}")
    fig.tight_layout()
    plt.show()

    unique_patterns = np.unique(patterns)
    bce_per_pattern = np.array([r["model_bce"][patterns == p].mean() for p in unique_patterns])
    mae_per_pattern = np.array([r["model_count_mae"][patterns == p].mean() for p in unique_patterns])

    x_axis = np.arange(len(unique_patterns))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x_axis, bce_per_pattern)
    ax.set_xticks(x_axis)
    ax.set_xticklabels(unique_patterns.astype(int).astype(str))
    ax.set_xlabel("Pattern")
    ax.set_ylabel("BCE")
    ax.set_title(f"BCE per pattern: {r['name']}")
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x_axis, mae_per_pattern)
    ax.set_xticks(x_axis)
    ax.set_xticklabels(unique_patterns.astype(int).astype(str))
    ax.set_xlabel("Pattern")
    ax.set_ylabel("Spike-count MAE")
    ax.set_title(f"Spike-count MAE per pattern: {r['name']}")
    fig.tight_layout()
    plt.show()

In [ ]:
# Visual plausibility check: true raster vs predicted probability map.
# Change selected_result and sample_id to inspect different recordings/samples.
selected_result = 0
sample_id = 0

r = results[selected_result]
x_true = r["Xval"][sample_id]
x_prob = r["predictions_val"][sample_id]

# Remove channel dimension for plotting.
x_true_plot = np.squeeze(x_true)
x_prob_plot = np.squeeze(x_prob)

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(x_true_plot, aspect="auto", interpolation="nearest")
ax.set_xlabel("Time bin")
ax.set_ylabel("Electrode / unit")
ax.set_title(f"True validation response raster: {r['name']}")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(x_prob_plot, aspect="auto", interpolation="nearest")
ax.set_xlabel("Time bin")
ax.set_ylabel("Electrode / unit")
ax.set_title(f"Predicted response probability map: {r['name']}")
fig.colorbar(im, ax=ax, label="p(spike)")
fig.tight_layout()
plt.show()

# Interpretation Note

This notebook answers a specific question: **do the recordings behave differently enough that they need separate models?**

Because each model is trained on exactly one `(network, DIV)` dataset and receives no network/DIV input, differences in outputs across recordings come from differences in the learned dataset-specific mapping, not from a recording-ID feature.

If the separate models produce similar predictions and metrics across recordings, then a pooled model without recording identity is worth trying next. If they differ substantially, pooling all recordings into one dataset may blur real biological differences unless the model includes some recording adaptation mechanism.
